In [7]:
import sys
sys.path.append('/host/d/Github')
import nibabel as nb
import glob
import os
import glob
import lpips
import torch
import numpy as np
import pandas as pd
from skimage.metrics import structural_similarity

import Diffusion_denoising_thin_slice.functions_collection as ff
import Diffusion_denoising_thin_slice.Build_lists.Build_list as Build_list
import Diffusion_denoising_thin_slice.Data_processing as Data_processing

In [8]:
build_sheet =  Build_list.Build_thinsliceCT(os.path.join('/host/e/D/Data/brain_CT/Patient_lists/fixedCT_static_simulation_train_test_gaussian_xjtlu.xlsx'))
_,patient_id_list,patient_subid_list,random_num_list, condition_list, x0_list = build_sheet.__build__(batch_list = [5]) 
n = ff.get_X_numbers_in_interval(total_number = patient_id_list.shape[0],start_number = 0,end_number = 1, interval = 2)

In [9]:
def calc_mae_with_ref_window(img, ref, vmin, vmax):
    maes = []
    for slice_num in range(0, img.shape[-1]):
        slice_img = img[:,:,slice_num]
        slice_ref = ref[:,:,slice_num]
        mask = np.where((slice_ref >= vmin) & (slice_ref <= vmax), 1, 0)
        mae = np.sum(np.abs(slice_img - slice_ref) * mask) / np.sum(mask)
        maes.append(mae)

    return np.mean(maes), np.std(maes)

In [10]:
def calc_ssim_with_ref_window(img, ref, vmin, vmax):

    ssims = []
    for slice_num in range(0, img.shape[-1]):
        slice_img = img[:,:,slice_num]
        slice_ref = ref[:,:,slice_num]
        mask = np.where((slice_ref >= vmin) & (slice_ref <= vmax), 1, 0)
        _, ssim_map = structural_similarity(slice_img, slice_ref, data_range=vmax - vmin, full=True)
        ssim = np.sum(ssim_map * mask) / np.sum(mask)
        ssims.append(ssim)

    return np.mean(ssims), np.std(ssims)

In [11]:
def calc_lpips(imgs1, imgs2, vmin, vmax):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    loss_fn = lpips.LPIPS().to(device)
    
    lpipss = []
    for slice_num in range(0, imgs1.shape[-1]):
        slice1 = imgs1[:,:,slice_num]
        slice2 = imgs2[:,:,slice_num]

        slice1 = np.clip(slice1, vmin, vmax).astype(np.float32)
        slice2 = np.clip(slice2, vmin, vmax).astype(np.float32)

        slice1 = (slice1 - vmin) / (vmax - vmin) * 2 - 1
        slice2 = (slice2 - vmin) / (vmax - vmin) * 2 - 1

        slice1 = np.stack([slice1, slice1, slice1], axis=-1)
        slice2 = np.stack([slice2, slice2, slice2], axis=-1)
        # print('after stack, slice1 shape:', slice1.shape, ' slice2 shape:', slice2.shape)

        slice1 = np.transpose(slice1, (2, 0, 1))[np.newaxis, ...]
        slice2 = np.transpose(slice2, (2, 0, 1))[np.newaxis, ...]
        # print('after transpose, slice1 shape:', slice1.shape, ' slice2 shape:', slice2.shape)

        slice1 = torch.from_numpy(slice1).to(device)
        slice2 = torch.from_numpy(slice2).to(device)

        lpips_val = loss_fn(slice1, slice2)
        lpipss.append(lpips_val.item())

      

    return np.mean(lpipss), np.std(lpipss)


### postprocess noise2score

In [13]:
## metric calculations
from genericpath import isfile
import shutil

results = []
save_collection = False

NFE_list = [30]
for i in range(0,n.shape[0]):
    patient_id = patient_id_list[n[i]]
    patient_subid = patient_subid_list[n[i]]
    random_n = random_num_list[n[i]]


    print(patient_id, patient_subid, random_n)

    save_folder = os.path.join('/host/d/projects/denoising/pred_collections/brain_CT', patient_id, patient_subid)
    ff.make_folder([os.path.dirname(save_folder), save_folder])

    # reference image
    gt_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_brainCT/pred_images', patient_id, patient_subid,'random_'+str(random_n), 'epoch61_1/gt_img.nii.gz')
    gt_img = nb.load(gt_file).get_fdata()
    # if save_collection:
    #     shutil.copy(gt_file, os.path.join(save_folder, 'gt_img.nii.gz'))

    # noisy image
    condition_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_brainCT/pred_images', patient_id, patient_subid,'random_'+str(random_n), 'epoch61_1/condition_img.nii.gz')
    condition_img = nb.load(condition_file).get_fdata()
    # if save_collection:
    #     shutil.copy(condition_file, os.path.join(save_folder, 'condition_img.nii.gz'))

    # noise2noise
    noise2noise_file = os.path.join('/host/d/projects/denoising/models/noise2noise_brainCT/pred_images', patient_id, patient_subid,'random_'+str(random_n), 'epoch78/pred_img.nii.gz')
    noise2noise_img = nb.load(noise2noise_file).get_fdata() 
    # if save_collection:
    #     shutil.copy(noise2noise_file, os.path.join(save_folder, 'noise2noise_img.nii.gz'))

    # noise2score
    noise2score_file = os.path.join('/host/d/projects/denoising/models/noise2score_brainCT/pred_images', patient_id, patient_subid,'random_'+str(random_n), 'epoch100/pred_img_postprocess.nii.gz')
    noise2score_img = nb.load(noise2score_file).get_fdata()
    # if save_collection:
    #     shutil.copy(noise2score_file, os.path.join(save_folder, 'noise2score_img.nii.gz'))

    # # DDM2_firststep
    # ddm2_first_file = os.path.join('/host/d/projects/denoising/models/DDM2_brainCT/pred_images', patient_id, patient_subid,'ddm2_first_step.nii.gz')
    # ddm2_first_img = nb.load(ddm2_first_file).get_fdata()
    # if save_collection:
    #     shutil.copy(ddm2_first_file, os.path.join(save_folder, 'ddm2_firststep_img.nii.gz'))

    # DDM2_finalstep
    ddm2_final_file = os.path.join('/host/d/projects/denoising/models/DDM2_brainCT/pred_images', patient_id, patient_subid,'ddm2_final.nii.gz')
    ddm2_final_img = nb.load(ddm2_final_file).get_fdata()
    # if save_collection:
    #     shutil.copy(ddm2_final_file, os.path.join(save_folder, 'ddm2_finalstep_img.nii.gz'))

    # supervised method
    supervised_file = os.path.join('/host/d/projects/denoising/models/supervised_poisson_brainCT/pred_images', patient_id, patient_subid,'random_'+str(random_n), 'epoch58_1/pred_img.nii.gz')
    supervised_img = nb.load(supervised_file).get_fdata() 
    # if save_collection:
    #     shutil.copy(supervised_file, os.path.join(save_folder, 'supervised_img.nii.gz'))

    ## calculate metrics
    # MAE
    vmin = 0; vmax = 100
    mae_condition, mae_condition_std = calc_mae_with_ref_window(condition_img, gt_img, vmin, vmax)
    mae_noise2noise, mae_noise2noise_std = calc_mae_with_ref_window(noise2noise_img, gt_img, vmin, vmax)
    mae_noise2score, mae_noise2score_std = calc_mae_with_ref_window(noise2score_img, gt_img, vmin, vmax)
    mae_ddm2_final, mae_ddm2_final_std = calc_mae_with_ref_window(ddm2_final_img, gt_img, vmin, vmax)
    mae_supervised, mae_supervised_std = calc_mae_with_ref_window(supervised_img, gt_img, vmin, vmax)

    # SSIM
    ssim_condition, ssim_condition_std = calc_ssim_with_ref_window(condition_img, gt_img, vmin, vmax)
    ssim_noise2noise, ssim_noise2noise_std = calc_ssim_with_ref_window(noise2noise_img, gt_img, vmin, vmax)
    ssim_noise2score, ssim_noise2score_std = calc_ssim_with_ref_window(noise2score_img, gt_img, vmin, vmax)
    ssim_ddm2_final, ssim_ddm2_final_std = calc_ssim_with_ref_window(ddm2_final_img, gt_img, vmin, vmax)
    ssim_supervised, ssim_supervised_std = calc_ssim_with_ref_window(supervised_img, gt_img, vmin, vmax)

    # LPIPS
    lpips_condition, _ = calc_lpips(condition_img, gt_img, vmin, vmax)
    lpips_noise2noise, _ = calc_lpips(noise2noise_img, gt_img, vmin, vmax)
    lpips_noise2score, _ = calc_lpips(noise2score_img, gt_img, vmin, vmax)
    lpips_ddm2_final, _ = calc_lpips(ddm2_final_img, gt_img, vmin, vmax)
    lpips_supervised, _ = calc_lpips(supervised_img, gt_img, vmin, vmax)

    per_patient_results = [patient_id, patient_subid, random_n, mae_condition, ssim_condition, lpips_condition, mae_noise2noise, ssim_noise2noise, lpips_noise2noise, mae_noise2score, ssim_noise2score, lpips_noise2score, mae_ddm2_final, ssim_ddm2_final, lpips_ddm2_final, mae_supervised, ssim_supervised, lpips_supervised]

    for NFE in NFE_list:
        print('NFE is: ', NFE)
        # our method (unsupervised), 1 inference
        unsupervised_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_brainCT_predict_noise/', 'pred_images_NFE'+str(NFE), patient_id, patient_subid,'random_'+str(random_n), 'epoch75_1/pred_img.nii.gz')
        unsupervised_img = nb.load(unsupervised_file).get_fdata()
        # if save_collection:
        #     shutil.copy(unsupervised_file, os.path.join(save_folder, 'unsupervised_img.nii.gz'))

        # our method (unsupervised), 10 inference
        unsupervised_avg10_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_brainCT_predict_noise/', 'pred_images_NFE'+str(NFE), patient_id, patient_subid,'random_'+str(random_n), 'epoch75avg/pred_img_scans10.nii.gz')
        unsupervised_avg10_img = nb.load(unsupervised_avg10_file).get_fdata()
        # if save_collection:
        #     shutil.copy(unsupervised_avg10_file, os.path.join(save_folder, 'unsupervised_avg10_img.nii.gz'))

        # our method (unsupervised), beta = 0, 20 inference
        unsupervised_avg20_file = os.path.join('/host/d/projects/denoising/models/unsupervised_gaussian_brainCT_predict_noise/', 'pred_images_NFE'+str(NFE), patient_id, patient_subid,'random_'+str(random_n), 'epoch75avg/pred_img_scans20.nii.gz')
        unsupervised_avg20_img = nb.load(unsupervised_avg20_file).get_fdata()
        # if save_collection:
        #     shutil.copy(unsupervised_avg20_file, os.path.join(save_folder, 'unsupervised_avg20_img.nii.gz'))

        mae_unsupervised, mae_unsupervised_std = calc_mae_with_ref_window(unsupervised_img, gt_img, vmin, vmax)
        mae_unsupervised_avg10, mae_unsupervised_avg10_std = calc_mae_with_ref_window(unsupervised_avg10_img, gt_img, vmin, vmax)
        mae_unsupervised_avg20, mae_unsupervised_avg20_std = calc_mae_with_ref_window(unsupervised_avg20_img, gt_img, vmin, vmax)
        
        ssim_unsupervised, ssim_unsupervised_std = calc_ssim_with_ref_window(unsupervised_img, gt_img, vmin, vmax)
        ssim_unsupervised_avg10, ssim_unsupervised_avg10_std = calc_ssim_with_ref_window(unsupervised_avg10_img, gt_img, vmin, vmax)
        ssim_unsupervised_avg20, ssim_unsupervised_avg20_std = calc_ssim_with_ref_window(unsupervised_avg20_img, gt_img, vmin, vmax)
        
        lpips_unsupervised, _ = calc_lpips(unsupervised_img, gt_img, vmin, vmax)
        lpips_unsupervised_avg10, _ = calc_lpips(unsupervised_avg10_img, gt_img, vmin, vmax)
        lpips_unsupervised_avg20, _ = calc_lpips(unsupervised_avg20_img, gt_img, vmin, vmax)

        print('NFE: ', NFE)
        print('mae_avg10, ssim_avg10, lpips_avg10: ', mae_unsupervised_avg10, ssim_unsupervised_avg10, lpips_unsupervised_avg10)
        print('mae_avg20, ssim_avg20, lpips_avg20: ', mae_unsupervised_avg20, ssim_unsupervised_avg20, lpips_unsupervised_avg20)

        per_patient_results = per_patient_results + [mae_unsupervised, ssim_unsupervised, lpips_unsupervised, mae_unsupervised_avg10, ssim_unsupervised_avg10, lpips_unsupervised_avg10, mae_unsupervised_avg20, ssim_unsupervised_avg20, lpips_unsupervised_avg20]

    results.append(per_patient_results)
    column_list = ['patient_id', 'patient_subid', 'random_n']
    method_list = ['condition', 'noise2noise', 'noise2score', 'ddm2_final', 'supervised']
    for NFE in NFE_list:
        method_list = method_list + ['unsupervised_NFE'+str(NFE), 'unsupervised_avg10_NFE'+str(NFE), 'unsupervised_avg20_NFE'+str(NFE)]
    metric_list = ['mae', 'ssim', 'lpips']
    for method in method_list:
        for metric in metric_list:
            column_list = column_list + [metric + '_' + method]

    dd = pd.DataFrame(results, columns = column_list)  
    dd.to_excel(os.path.join('/host/d/projects/denoising/results', 'brainCT_results_NFE30.xlsx'), index = False)

00214841 0000455418 0
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
NFE is:  30
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Settin

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
NFE is:  30
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/loc

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
NFE is:  30
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/loc

...

Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
NFE is:  30
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/loc

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
NFE is:  30
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/loc

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
NFE is:  30
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/loc